# Clase 8 — Práctica: análisis del modelo

Curso "Deep Learning con Python y PyTorch desde cero" — DIH DATAlife (Hub de
Innovación Digital Europeo)

En esta práctica trabajas con dos ejercicios sobre un modelo ya entrenado
para reconocer dígitos escritos a mano del dataset MNIST (Modified National
Institute of Standards and Technology):

1. Completar el cálculo de la precisión (accuracy) sobre el conjunto de test
   a partir de las predicciones del modelo
2. Guardar los pesos del modelo con `torch.save`, "olvidarlo" creando un
   modelo nuevo sin entrenar, y recargar los pesos guardados con
   `torch.load` y `load_state_dict`

Completa las celdas marcadas con `# TODO`. Cuando termines cada ejercicio,
despliega la solución para comprobar tu respuesta, pero intenta resolverlo
por tu cuenta antes de mirarla.

## Preparación común

Ejecuta esta celda antes de empezar cualquiera de los dos ejercicios: carga
MNIST, define un modelo pequeño y lo entrena unas pocas épocas para tener un
modelo "ya entrenado" con el que trabajar en los dos ejercicios. No hace
falta que entiendas cada línea de esta celda: ya viste este proceso en
clases anteriores, y en esta práctica nos interesa el análisis, no el
entrenamiento.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

transform = transforms.ToTensor()
train_dataset_completo = torchvision.datasets.MNIST(
    root="./datos", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(
    root="./datos", train=False, download=True, transform=transform)

N_TRAIN = 3000
train_dataset = Subset(train_dataset_completo, list(range(N_TRAIN)))

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


class RedDensa(nn.Module):
    def __init__(self):
        super().__init__()
        self.aplanar = nn.Flatten()
        self.capa1 = nn.Linear(28 * 28, 256)
        self.capa2 = nn.Linear(256, 128)
        self.capa_salida = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.aplanar(x)
        x = self.relu(self.capa1(x))
        x = self.relu(self.capa2(x))
        return self.capa_salida(x)


def entrenar(modelo, data_loader, num_epochs):
    funcion_perdida = nn.CrossEntropyLoss()
    optimizador = optim.Adam(modelo.parameters(), lr=1e-3)
    modelo.train()
    for _ in range(num_epochs):
        for imagenes, etiquetas in data_loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            optimizador.zero_grad()
            salidas = modelo(imagenes)
            perdida = funcion_perdida(salidas, etiquetas)
            perdida.backward()
            optimizador.step()


net = RedDensa().to(device)
entrenar(net, train_loader, num_epochs=5)
net.eval()
print("Modelo entrenado y listo para analizar.")

## Ejercicio 1 — Calcula la precisión sobre el conjunto de test

Ya tienes el modelo `net` entrenado. Completa la celda siguiente para
calcular su precisión sobre `test_loader`, el conjunto de test que el
modelo no ha visto durante el entrenamiento.

**Pistas**

- `torch.argmax(salidas, dim=1)` te da, para cada imagen del lote, el índice
  de la clase con el valor más alto: la predicción del modelo
- Comparar `predicciones == etiquetas` te da un tensor de valores
  booleanos (`True`/`False`); `.sum().item()` cuenta cuántos son `True`
- La precisión es el número de aciertos dividido entre el número total de
  ejemplos evaluados
- Recuerda envolver el bucle en `torch.no_grad()`: solo estás evaluando, no
  entrenando

In [ ]:
aciertos = 0
total = 0

net.eval()
with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        salidas = net(imagenes)

        # TODO: calcula 'predicciones' a partir de 'salidas' usando
        # torch.argmax (la clase con el valor más alto, dim=1)
        predicciones = None  # TODO

        # TODO: suma a 'aciertos' el número de predicciones correctas de
        # este lote (compara 'predicciones' con 'etiquetas')
        # TODO: suma a 'total' el número de ejemplos de este lote
        # (etiquetas.size(0))

precision = None  # TODO: aciertos / total
print(f"Precisión sobre el conjunto de test: {precision:.2%}")

<details>
<summary>💡 Ver solución</summary>

```python
aciertos = 0
total = 0

net.eval()
with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        salidas = net(imagenes)

        # La clase predicha es el índice con el valor más alto en la salida
        predicciones = torch.argmax(salidas, dim=1)

        aciertos += (predicciones == etiquetas).sum().item()
        total += etiquetas.size(0)

precision = aciertos / total
print(f"Precisión sobre el conjunto de test: {precision:.2%}")

# Con un modelo entrenado unas pocas épocas sobre 3000 imágenes, la
# precisión suele rondar el 90%: nada mal para un modelo tan sencillo,
# aunque un modelo entrenado con las 60 000 imágenes completas llegaría
# más alto todavía.
```
</details>

## Ejercicio 2 — Guarda los pesos, "olvida" el modelo y recárgalos

Vas a comprobar en primera persona por qué `torch.save` es tan útil: guarda
los pesos de `net` en un archivo, crea después un modelo **nuevo y sin
entrenar** (como si hubieras cerrado el notebook y perdido `net`), y
recupera en él los pesos guardados.

Completa los `# TODO`:

1. Guarda los pesos de `net` (su `state_dict`) en la ruta
   `ejercicio2_pesos.pt` con `torch.save`
2. Crea una instancia nueva de `RedDensa` llamada `modelo_nuevo` y muévela
   al mismo `device`
3. Antes de cargar los pesos, calcula la precisión de `modelo_nuevo` sobre
   `test_loader` (reutiliza la lógica del ejercicio 1) y guárdala en
   `precision_antes`
4. Carga los pesos guardados en `modelo_nuevo` con `load_state_dict` y
   `torch.load`
5. Vuelve a calcular la precisión y guárdala en `precision_despues`

**Pistas**

- `modelo_nuevo.load_state_dict(torch.load(ruta, map_location=device))` es
  la línea clave para recargar los pesos
- Antes de cargar los pesos, la precisión debería rondar el 10% (equivalente
  a elegir un dígito al azar entre diez posibles)
- Después de cargar los pesos, la precisión debería coincidir con la que
  obtuviste en el ejercicio 1

In [ ]:
def calcular_precision(modelo, data_loader):
    aciertos = 0
    total = 0
    modelo.eval()
    with torch.no_grad():
        for imagenes, etiquetas in data_loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            predicciones = torch.argmax(salidas, dim=1)
            aciertos += (predicciones == etiquetas).sum().item()
            total += etiquetas.size(0)
    return aciertos / total


ruta_pesos = "ejercicio2_pesos.pt"

# TODO 1: guarda los pesos de 'net' (net.state_dict()) en 'ruta_pesos'
# con torch.save


# TODO 2: crea una instancia nueva de RedDensa, sin entrenar, y muévela
# al mismo 'device' que 'net'
modelo_nuevo = None  # TODO

# TODO 3: calcula la precisión de 'modelo_nuevo' ANTES de cargar los pesos
# usando calcular_precision(modelo_nuevo, test_loader)
precision_antes = None  # TODO
print(f"Precisión antes de cargar los pesos: {precision_antes:.2%}")

# TODO 4: carga en 'modelo_nuevo' los pesos guardados en 'ruta_pesos' con
# modelo_nuevo.load_state_dict(torch.load(...))


# TODO 5: calcula la precisión de 'modelo_nuevo' DESPUÉS de cargar los pesos
precision_despues = None  # TODO
print(f"Precisión después de cargar los pesos: {precision_despues:.2%}")

<details>
<summary>💡 Ver solución</summary>

```python
def calcular_precision(modelo, data_loader):
    aciertos = 0
    total = 0
    modelo.eval()
    with torch.no_grad():
        for imagenes, etiquetas in data_loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            predicciones = torch.argmax(salidas, dim=1)
            aciertos += (predicciones == etiquetas).sum().item()
            total += etiquetas.size(0)
    return aciertos / total


ruta_pesos = "ejercicio2_pesos.pt"

# Guardamos únicamente los pesos aprendidos de 'net'
torch.save(net.state_dict(), ruta_pesos)

# Modelo nuevo, con pesos inicializados al azar: es como si 'net' nunca
# hubiera existido
modelo_nuevo = RedDensa().to(device)

precision_antes = calcular_precision(modelo_nuevo, test_loader)
print(f"Precisión antes de cargar los pesos: {precision_antes:.2%}")

# Cargamos los pesos guardados dentro de la arquitectura nueva
modelo_nuevo.load_state_dict(torch.load(ruta_pesos, map_location=device))

precision_despues = calcular_precision(modelo_nuevo, test_loader)
print(f"Precisión después de cargar los pesos: {precision_despues:.2%}")

# precision_antes ronda el 10% (pesos aleatorios, equivalente a adivinar
# entre diez dígitos). precision_despues coincide con la precisión del
# ejercicio 1: hemos recuperado el conocimiento aprendido por 'net' sin
# volver a entrenar ni un solo paso.
```
</details>

## Para seguir practicando

Si quieres ir un poco más allá:

- Calcula también la matriz de confusión sobre `test_loader` (como en el
  notebook de teoría) y busca qué par de dígitos confunde más tu modelo
- En el ejercicio 2, intenta cargar los pesos guardados en una arquitectura
  ligeramente distinta (por ejemplo, cambiando el tamaño de `capa1` de 256 a
  300 neuronas) y observa el error que PyTorch lanza: es la forma más
  directa de entender por qué la arquitectura debe coincidir exactamente
- Entrena `net` unas pocas épocas más sobre las 3000 imágenes y comprueba si
  empieza a sobreajustar: compara su precisión sobre `train_loader` frente a
  `test_loader`

Repasa el capítulo 08 del manual si necesitas refrescar la teoría antes de
intentarlo.